# Assignment 2: Link prediction with supervised learning

**Pipeline (follows Lab 5):** import -> hide some edges -> explore -> features -> labelled pairs (positives + negatives) -> model selection -> predict `solutionInput.csv` -> write output -> evaluate.

**Key idea used throughout:** a pair's features must be computed on a graph that does **not** contain that pair's edge, because that is exactly the situation of the real test links (they are missing from the network we can see). Training positives are therefore scored on fold graphs `H` that hide them; validation/test positives were already removed from `T` in section 2.

| Section | Content |
|---|---|
| 1 | Import and checks |
| 2 | Hide edges: training / validation / test |
| 3 | Explore the network (attributes, communities) |
| 4 | Feature engineering |
| 5 | Build datasets |
| 6 | Model selection and tuning |
| 7 | Predict `solutionInput.csv` |
| 8 | Output file and final evaluation |
| 9 | TODO: ideas to improve performance |
| 10 | Notes for the report |

## 1. Import the network data

In [98]:
import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from networkx.algorithms.community import greedy_modularity_communities
from networkx.algorithms.community.quality import modularity

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score)
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
np.random.seed(SEED)   # repeatable random negatives

In [99]:
G = nx.read_edgelist("input/edges_train.edgelist", delimiter=",",
                     nodetype=int, create_using=nx.Graph())
atts = pd.read_csv("input/attributes.csv")

print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")
print("Graph nodes without an attribute:", len(set(G.nodes) - set(atts["ID"])))
print("Attribute rows without a graph node:", len(set(atts["ID"]) - set(G.nodes)))
print(atts["attribute"].value_counts().sort_index().to_dict())

Nodes: 1800, Edges: 6377
Graph nodes without an attribute: 0
Attribute rows without a graph node: 0
{'a': 300, 'b': 300, 'c': 300, 'd': 300, 'e': 300, 'f': 300}


## 2. Hide some edges (training / validation / test)

`T` is the graph the model is allowed to see: it contains **only the training edges**. Validation and test edges are real edges deliberately hidden from `T`, just like the real missing links are hidden from `G`.

| Group | Share | Role |
|---|---|---|
| training edges | 70% | teach the model |
| validation edges | 15% | choose features / models |
| test edges | 15% | final score (use once, in section 8) |

In [100]:
edge_list = np.array(G.edges)
training_edges, temp_edges = train_test_split(edge_list, test_size=0.3, random_state=SEED)
validation_edges, test_edges = train_test_split(temp_edges, test_size=0.5, random_state=SEED)

def new_graph(edges, main):
    """Graph with the given edges and ALL nodes of `main` (keeps nodes without edges)."""
    H = nx.Graph()
    H.add_nodes_from(main.nodes())
    H.add_edges_from(edges)
    return H

T = new_graph(training_edges, G)

print(len(training_edges), len(validation_edges), len(test_edges))
print("val edges inside T: ", sum(T.has_edge(u, v) for u, v in validation_edges))
print("test edges inside T:", sum(T.has_edge(u, v) for u, v in test_edges))

4463 957 957
val edges inside T:  0
test edges inside T: 0


## 3. Explore the network (attributes and communities)

Exploration only: nothing here feeds the model except the idea that **communities line up with attributes**, which is why `same_community` becomes a feature in section 4.

In [101]:
# Communities on the full visible graph (greedy modularity)
c = list(greedy_modularity_communities(G, resolution=1))
print(f"Communities: {len(c)} | modularity: {modularity(G, c):.3f}")
print("Sizes:", sorted((len(x) for x in c), reverse=True))


Communities: 6 | modularity: 0.742
Sizes: [306, 303, 301, 299, 299, 292]


In [102]:
# Same layout for both plots so they can be compared
pos = nx.spring_layout(G, seed=SEED)

atts["attribute_code"] = pd.Categorical(atts["attribute"]).codes
attr_code = atts.set_index("ID")["attribute_code"].to_dict()
node_to_comm = {n: i for i, comm in enumerate(c) for n in comm}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, colours, title in [
    (axes[0], [attr_code.get(n, -1) for n in G.nodes()], "Coloured by attribute"),
    (axes[1], [node_to_comm[n] for n in G.nodes()],       f"Coloured by community (n = {len(c)})"),
]:
    nx.draw_networkx_edges(G, pos, ax=ax, edge_color="lightgray", alpha=0.35, width=0.4)
    nx.draw_networkx_nodes(G, pos, ax=ax, node_color=colours, cmap=plt.cm.tab10, node_size=20)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

/var/folders/d2/xd2z0dzd25906v020j08jqd00000gn/T/ipykernel_70413/711471647.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [103]:
# Do communities (found on the training graph T only) match attributes?
comms_T = nx.community.louvain_communities(T, seed=SEED)
print(len(comms_T), "communities; the tiny ones are nodes with no edge in T")
comm_of_T = {n: i for i, comm in enumerate(comms_T) for n in comm}
print(pd.crosstab(pd.Series(comm_of_T, name="community"),
                  atts.set_index("ID")["attribute"]).loc[lambda d: d.sum(axis=1) > 10])

# How strongly do edges prefer same-class endpoints?
attr = atts.set_index("ID")["attribute"]
same = np.mean([attr[u] == attr[v] for u, v in training_edges])
print(f"Share of training edges joining same-class nodes: {same:.2f} (about 0.17 if class did not matter)")

18 communities; the tiny ones are nodes with no edge in T
attribute    a    b    c    d    e    f
community                              
2           13   12   20   13   17  217
3           26  216   19   20   23   10
4          222   15   17   17   11   14
7           10   15  210   15   19   18
8           13   20   19   21  210   20
13          16   18   14  213   16   19
Share of training edges joining same-class nodes: 0.54 (about 0.17 if class did not matter)


## 4. Feature engineering
*(Lab 5: `getFeature`, Q3)*

Every feature function takes the **graph as an argument**, so the same code can run on `H` (training folds), `T` (validation/test) or `G` (final predictions). Families:

- **4a** NetworkX similarity measures
- **4b** simple measures (degrees, common neighbours)
- **4c** node attribute (homophily)
- **4d** community membership (Louvain, recomputed on whichever graph is passed in)

### 4a. NetworkX similarity measures

In [104]:
# name -> networkx function (each column of the final table comes from here)
topology_funcs = {
    "jaccard":        nx.jaccard_coefficient,
    "adamic_adar":    nx.adamic_adar_index,
    "resource_alloc": nx.resource_allocation_index,
    "pref_attach":    nx.preferential_attachment,
    "ccpa":           nx.common_neighbor_centrality,
}

def edge_scores(func, G, pairs, **kwargs):
    """Run a networkx link-prediction function over pairs; scores come back in the same order."""
    return [score for _, _, score in func(G, pairs, **kwargs)]

### 4b. Simple measures

In [105]:
def simple_features(pairs, G):
    rows = []
    for u, v in pairs:
        nu, nv = set(G.neighbors(u)), set(G.neighbors(v))
        rows.append({
            "degree_u": G.degree(u),
            "degree_v": G.degree(v),
            "common_neighbours": len(nu & nv),
            # TODO (section 9): log-degree product, min/max degree, clustering coefficient ...
        })
    return pd.DataFrame(rows)

### 4c. Node attribute

In [106]:
def attribute_features(pairs, atts):
    attr_by_id = atts.set_index("ID")["attribute"]
    return pd.DataFrame({"same_attribute": [int(attr_by_id[u] == attr_by_id[v]) for u, v in pairs]})

### 4d. Community membership

In [107]:
def community_features(pairs, G):
    """1 if both nodes land in the same Louvain community of the graph G passed in."""
    comms = nx.community.louvain_communities(G, seed=SEED)
    comm_of = {n: i for i, comm in enumerate(comms) for n in comm}
    return pd.DataFrame({"same_community": [int(comm_of[u] == comm_of[v]) for u, v in pairs]})

### Assemble everything

In [108]:
def create_network_features(pairs, G, feature_funcs=topology_funcs):
    """One row per pair, one column per feature. G = the graph the features are computed on."""
    pairs = [tuple(p) for p in pairs]
    parts = [simple_features(pairs, G)]
    parts.append(pd.DataFrame({name: edge_scores(func, G, pairs) for name, func in feature_funcs.items()}))
    parts.append(attribute_features(pairs, atts))
    parts.append(community_features(pairs, G))
    return pd.concat(parts, axis=1)

## 5. Build the datasets
*(Lab 5: Q4, how many negatives?)*

- `make_dataset` turns a list of **real edges** into labelled pairs: the edges (label 1) plus the same number of random non-edges (label 0). Negatives are checked against the full graph `G`, so a real hidden link is never labelled 0.
- `build_train_features` computes training features in **folds**. For each fold, the graph `H` omits that fold's edges, the fold's edges are the positives, and features come from `H`. No positive ever sees its own edge.
- Validation (and test) positives were already removed from `T`, so their features are computed directly on `T`.

In [ ]:
def make_dataset(edge_list, G, ratio=1):
    sorted_positives = [tuple(sorted(map(int, e))) for e in edge_list]   # sorted like the negatives
    nodes = list(G.nodes)
    n_negatives = int(len(sorted_positives) * ratio)

    negative_set = set()
    while len(negative_set) < n_negatives:
        i = np.random.randint(len(nodes))
        j = np.random.randint(len(nodes))
        pair = tuple(sorted((i, j)))

        while G.has_edge(i, j) or i == j or pair in negative_set:
            i = np.random.randint(len(nodes))
            j = np.random.randint(len(nodes))
            pair = tuple(sorted((i, j)))

        negative_set.add(pair)

    X = list(sorted_positives) + list(negative_set)
    y = [1] * len(sorted_positives) + [0] * len(negative_set)
    return X, y



In [ ]:
def build_train_features(training_edges, T, G, n_folds=5, ratio=1):
    """
    Build training features WITHOUT data leakage.

    Problem: training positives are edges of T. If a pair's features are computed on a
    graph that still contains its own edge, the features contain the answer.
      - degree_u / degree_v are one higher than they would be for a missing link
      - ccpa depends on distance, and a linked pair has distance 1, so ccpa is 360 for a
        linked pair (with no shared friends) and far lower for any unlinked pair
    The model then learns "ccpa = 360 means link" (train accuracy 1.00, validation 0.50).
    The real test links are missing from the network, so that signal never appears there.

    Fix: score every training positive on a graph where its edge has been hidden.
    K-fold gives us that without recomputing the graph once per edge.
    """
    parts, labels = [], []

    # Split the training EDGES (not the pairs) into n_folds groups.
    # shuffle + random_state: random assignment, but identical on every run.
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=SEED)

    for fit_idx, hold_idx in kf.split(training_edges):
        # fit_idx  = ~80% of the training edges: these are allowed to build the graph
        # hold_idx = ~20% of the training edges: these become this round's positives

        # Feature graph for this fold: everything EXCEPT the held-out edges.
        # None of the held-out edges exist in H, so none of them can influence
        # their own features (degree, distance, shared friends, community).
        H = new_graph(training_edges[fit_idx], T)

        # Labelled pairs for this fold: the held-out edges (label 1) plus an equal
        # number of random non-edges (label 0). Negatives are checked against the FULL
        # graph G, so a real link is never mislabelled as 0. No features are computed
        # here, so using G at this point leaks nothing.
        pairs, y = make_dataset(training_edges[hold_idx], G, ratio)

        # Features come from H, never from T or G. Positives look like missing links
        # (their edge is absent). Negatives look the same, so both classes are measured
        # under identical conditions in each fold. Louvain communities are also
        # recomputed on H, so the community feature cannot "see" the held-out edges either.
        parts.append(create_network_features(pairs, H))
        labels.extend(y)

    # Every training edge was held out exactly once, so every positive has a
    # leak-free feature row. Stack the folds into one table (about 8,900 rows).
    return pd.concat(parts, ignore_index=True), np.array(labels)

In [110]:
np.random.seed(SEED)   # same negatives on every run

X_train_features, y_train = build_train_features(training_edges, T, G)

X_val_pairs, y_val = make_dataset(validation_edges, G, 1)
X_val_features = create_network_features(X_val_pairs, T)
y_val = np.array(y_val)

# The test set is built in section 8, so no model choice can be influenced by it.

In [111]:
# Sanity checks
print(X_train_features.shape, len(y_train), "| share positive:", y_train.mean())
print(X_val_features.shape, len(y_val), "| share positive:", y_val.mean())
print("same columns:", list(X_train_features.columns) == list(X_val_features.columns))
print("NaN:", int(X_train_features.isna().sum().sum()), "| inf:", int(np.isinf(X_train_features).sum().sum()))

corr = X_train_features.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
print("\nHighly correlated pairs (>0.9):")
print(upper.stack().sort_values(ascending=False).loc[lambda s: s > 0.9])

(8926, 10) 8926 | share positive: 0.5
(1914, 10) 1914 | share positive: 0.5
same columns: True
NaN: 0 | inf: 0

Highly correlated pairs (>0.9):
adamic_adar        resource_alloc    0.980194
common_neighbours  adamic_adar       0.921785
dtype: float64


## 6. Model selection and tuning
*(Lab 5: Q5)*

Rules: choose features and models using **training and validation only**. Differences smaller than about 0.015 in validation accuracy are within noise (finite validation set plus random negatives), so prefer the simpler option when scores tie.

### 6a. Baseline: logistic regression on all features

In [112]:
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
lr.fit(X_train_features, y_train)

dummy = DummyClassifier(strategy="most_frequent").fit(X_train_features, y_train)

print("train accuracy:", round(accuracy_score(y_train, lr.predict(X_train_features)), 4))
print("val accuracy:  ", round(accuracy_score(y_val, lr.predict(X_val_features)), 4))
print("baseline:      ", round(accuracy_score(y_val, dummy.predict(X_val_features)), 4))

coefs = pd.Series(lr[-1].coef_[0], index=X_train_features.columns)
print("\nScaled coefficients (do not read the sign of correlated features individually):")
print(coefs.sort_values(key=abs, ascending=False).round(3))

train accuracy: 0.8172
val accuracy:   0.8464
baseline:       0.5

Scaled coefficients (do not read the sign of correlated features individually):
same_community       1.271
degree_v             0.755
degree_u             0.694
same_attribute       0.520
pref_attach         -0.345
ccpa                -0.294
resource_alloc       0.294
adamic_adar          0.260
common_neighbours    0.233
jaccard              0.058
dtype: float64


### 6b. Which features matter?
- **alone**: validation accuracy of a model using only this feature.
- **drop_loss**: how much accuracy is lost when the feature is removed from the full model.

High alone + ~0 drop_loss means useful but duplicated by another feature.

In [113]:
def val_acc(model, cols):
    model.fit(X_train_features[cols], y_train)
    return accuracy_score(y_val, model.predict(X_val_features[cols]))

cols_all = list(X_train_features.columns)
full = val_acc(lr, cols_all)

rows = [{"feature": c,
         "alone": val_acc(lr, [c]),
         "drop_loss": full - val_acc(lr, [x for x in cols_all if x != c])} for c in cols_all]

print("all features:", round(full, 4), "| random guessing: 0.5")
print(pd.DataFrame(rows).sort_values("alone", ascending=False).round(4).to_string(index=False))

all features: 0.8464 | random guessing: 0.5
          feature  alone  drop_loss
   same_community 0.8417     0.0632
             ccpa 0.7602     0.0005
   same_attribute 0.6938     0.0000
      pref_attach 0.6223     0.0037
common_neighbours 0.6196     0.0000
          jaccard 0.6196     0.0000
      adamic_adar 0.6196     0.0000
   resource_alloc 0.6196     0.0000
         degree_u 0.6176     0.0063
         degree_v 0.5972     0.0010


### 6c. Candidate feature sets

In [114]:
chosen = ["same_community", "ccpa", "same_attribute"]   # TODO: revisit after trying the ideas in section 9

cands = {
    "same_community only":                    ["same_community"],
    "community + degrees":                    ["same_community", "degree_u", "degree_v"],
    "community + ccpa + attribute":           chosen,
    "community + ccpa + attribute + degrees": chosen + ["degree_u", "degree_v"],
    "all features":                           cols_all,
}
for name, cols in cands.items():
    print(f"{name:42s} {val_acc(lr, cols):.4f}  ({len(cols)} features)")

same_community only                        0.8417  (1 features)
community + degrees                        0.8433  (3 features)
community + ccpa + attribute               0.8459  (3 features)
community + ccpa + attribute + degrees     0.8412  (5 features)
all features                               0.8464  (10 features)


### 6d. Logistic regression vs random forest

In [115]:
rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=SEED, n_jobs=-1)

for set_name, cols in {"chosen": chosen, "all": cols_all}.items():
    for m_name, model in {"LR": lr, "RF": rf}.items():
        model.fit(X_train_features[cols], y_train)
        tr = accuracy_score(y_train, model.predict(X_train_features[cols]))
        va = accuracy_score(y_val,   model.predict(X_val_features[cols]))
        print(f"{m_name}  {set_name:7s} train {tr:.4f}  val {va:.4f}")

LR  chosen  train 0.8122  val 0.8459
RF  chosen  train 0.8274  val 0.8485
LR  all     train 0.8172  val 0.8464
RF  all     train 0.8474  val 0.8485


### 6e. (Optional) How big is the noise?
Redraw the validation negatives with 10 different seeds and compare LR and RF each time. If `RF_minus_LR` flips sign, the models are tied. Takes about a minute.

In [116]:
lr_fit = clone(lr).fit(X_train_features[chosen], y_train)
rf_fit = RandomForestClassifier(n_estimators=300, max_depth=5, min_samples_leaf=20,
                                random_state=SEED, n_jobs=-1).fit(X_train_features[chosen], y_train)

res = []
for seed in range(10):
    np.random.seed(seed)
    pairs, y = make_dataset(validation_edges, G, 1)
    F = create_network_features(pairs, T)[chosen]
    res.append({"LR": accuracy_score(y, lr_fit.predict(F)), "RF": accuracy_score(y, rf_fit.predict(F))})
np.random.seed(SEED)

res = pd.DataFrame(res)
res["RF_minus_LR"] = res["RF"] - res["LR"]
print(res.round(4))
print(res.agg(["mean", "std"]).round(4))

       LR      RF  RF_minus_LR
0  0.8370  0.8396       0.0026
1  0.8328  0.8380       0.0052
2  0.8422  0.8422       0.0000
3  0.8359  0.8396       0.0037
4  0.8401  0.8438       0.0037
5  0.8542  0.8589       0.0047
6  0.8412  0.8391      -0.0021
7  0.8412  0.8401      -0.0010
8  0.8333  0.8365       0.0031
9  0.8401  0.8433       0.0031
          LR      RF  RF_minus_LR
mean  0.8398  0.8421       0.0023
std   0.0061  0.0063       0.0025


### 6f. Hyperparameter tuning (training data only, cross-validated)
Note: the same node appears in several CV folds, so CV scores are slightly optimistic. They are still fine for comparing settings.

In [117]:
grid_lr = GridSearchCV(
    make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    param_grid={"logisticregression__C": [0.001, 0.01, 0.1, 1, 10, 100]},
    cv=5, scoring="accuracy")
grid_lr.fit(X_train_features[chosen], y_train)
print("LR :", grid_lr.best_params_, round(grid_lr.best_score_, 4))

grid_rf = GridSearchCV(
    RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
    param_grid={"max_depth": [3, 5, 10, None], "min_samples_leaf": [1, 5, 20]},
    cv=5, scoring="accuracy")
grid_rf.fit(X_train_features[chosen], y_train)
print("RF :", grid_rf.best_params_, round(grid_rf.best_score_, 4))

print(pd.DataFrame(grid_lr.cv_results_)[["param_logisticregression__C", "mean_test_score"]].round(4))

LR : {'logisticregression__C': 0.001} 0.8122
RF : {'max_depth': 5, 'min_samples_leaf': 5} 0.8274
   param_logisticregression__C  mean_test_score
0                        0.001           0.8122
1                        0.010           0.8122
2                        0.100           0.8104
3                        1.000           0.8104
4                       10.000           0.8104
5                      100.000           0.8104


### 6g. Lock in the final model
Set the final features and model here. Everything after this uses `final_model` and `FINAL_COLS`. Default: logistic regression on the 3 chosen features with the tuned `C` (simple and explainable). Switch to the random forest only if section 6e shows it winning consistently.

In [118]:
FINAL_COLS = chosen

best_C = grid_lr.best_params_["logisticregression__C"]
final_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, C=best_C))
# Alternative (only if 6e favours it):
# final_model = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1, **grid_rf.best_params_)

final_model.fit(X_train_features[FINAL_COLS], y_train)
print("train accuracy:", round(accuracy_score(y_train, final_model.predict(X_train_features[FINAL_COLS])), 4))
print("val accuracy:  ", round(accuracy_score(y_val,   final_model.predict(X_val_features[FINAL_COLS])), 4))

train accuracy: 0.8122
val accuracy:   0.8459


### 6h. Where are the errors? (validation)

In [119]:
pred  = final_model.predict(X_val_features[FINAL_COLS])
proba = final_model.predict_proba(X_val_features[FINAL_COLS])[:, 1]

print(confusion_matrix(y_val, pred))
print(classification_report(y_val, pred, digits=3))
print("AUC:", round(roc_auc_score(y_val, proba), 4))

d = X_val_features[FINAL_COLS].assign(label=y_val, pred=pred)
print(pd.crosstab([d.same_community, d.label], d.pred))
print("\nRows: same_community (0/1) and true label. Which kind of mistake dominates?")

[[806 151]
 [144 813]]
              precision    recall  f1-score   support

           0      0.848     0.842     0.845       957
           1      0.843     0.850     0.846       957

    accuracy                          0.846      1914
   macro avg      0.846     0.846     0.846      1914
weighted avg      0.846     0.846     0.846      1914

AUC: 0.8837
pred                    0    1
same_community label          
0              0      806    7
               1      144   15
1              0        0  144
               1        0  798

Rows: same_community (0/1) and true label. Which kind of mistake dominates?


## 7. Predict the links in `solutionInput.csv`
*(Lab 5: Q6, "Return Solution")*

### 7a. Load the input pairs

In [120]:
inp_df = pd.read_csv("input/solutionInput.csv", sep=",", index_col="ID")
assert inp_df.shape[1] == 2, "expected two node columns"
inp_pairs = [tuple(sorted(map(int, row))) for row in inp_df.values]

print(inp_df.shape)
print("nodes not in G:", len({n for p in inp_pairs for n in p} - set(G.nodes)))

(1416, 2)
nodes not in G: 0


### 7b. Which graph should the features come from: `T` or `G`?

The model was trained on features from sparser graphs (`H`, then validated on `T`). `G` has more edges than either, so the choice matters. Two checks:

1. **With labels (validation):** score the same validation pairs with features from a graph at *training* density (`H80`) and from the denser `T`. If denser is not worse, a denser graph for the real pairs is reasonable.
2. **Without labels (the real file):** the input file is exactly 50% links, so a good setup should predict about 50% positive.

In [121]:
H80_edges, _ = train_test_split(training_edges, train_size=0.8, random_state=0)
H80 = new_graph(H80_edges, T)

for name, graph in [("H80 (training density)", H80), ("T (denser)", T)]:
    F = create_network_features(X_val_pairs, graph)[FINAL_COLS]
    print(f"val accuracy, features from {name:24s}", round(accuracy_score(y_val, final_model.predict(F)), 4))

val accuracy, features from H80 (training density)   0.8145
val accuracy, features from T (denser)               0.8459


In [122]:
feat_T = create_network_features(inp_pairs, T)[FINAL_COLS]
feat_G = create_network_features(inp_pairs, G)[FINAL_COLS]
pred_T, pred_G = final_model.predict(feat_T), final_model.predict(feat_G)

print("share predicted 'link' (true share is 0.50):  T:", round(pred_T.mean(), 3), "| G:", round(pred_G.mean(), 3))
print("predictions that agree between T and G:      ", round((pred_T == pred_G).mean(), 3))

share predicted 'link' (true share is 0.50):  T: 0.523 | G: 0.553
predictions that agree between T and G:       0.947


### 7c. Final predictions
`FEATURE_GRAPH = G` by default (most information, closest to the original network). Switch it to `T` if the checks above favour `T` (for example, if `G` gives a predicted share far from 0.50 or validation accuracy drops with denser graphs).

In [123]:
FEATURE_GRAPH = G   # TODO: confirm using the checks in 7b

X_inp_features = create_network_features(inp_pairs, FEATURE_GRAPH)[FINAL_COLS]
predictions = final_model.predict(X_inp_features)

print(len(predictions), "predictions | share predicted 'link':", round(predictions.mean(), 3))

1416 predictions | share predicted 'link': 0.553


## 8. Output file and evaluation
*(Lab 5: "Output solution" and "Evaluate solution")*

### 8a. Write the predictions
Same format as Lab 5: columns `ID` and `prediction` (1 = link exists, 0 = it does not).

In [124]:
out = pd.DataFrame({"prediction": predictions.astype(int)}, index=inp_df.index)
out.to_csv("predictions.csv", index=True, header=["prediction"], index_label="ID")

check = pd.read_csv("predictions.csv", index_col="ID")
print(check.shape, "| values:", sorted(check["prediction"].unique()))
print(check.head())

(1416, 1) | values: [np.int64(0), np.int64(1)]
    prediction
ID            
0            1
1            1
2            1
3            1
4            0


### 8b. Final evaluation on the held-out test set
This is the one place the test set is used. Do not change features or models after looking at this number. (State in the report if the test set was looked at earlier during development.)

In [125]:
np.random.seed(SEED)
X_test_pairs, y_test = make_dataset(test_edges, G, 1)
y_test = np.array(y_test)
X_test_features = create_network_features(X_test_pairs, T)

test_pred  = final_model.predict(X_test_features[FINAL_COLS])
test_proba = final_model.predict_proba(X_test_features[FINAL_COLS])[:, 1]

print("test accuracy:", round(accuracy_score(y_test, test_pred), 4), "| random / always-'no link' baseline: 0.5")
print(confusion_matrix(y_test, test_pred))
print(classification_report(y_test, test_pred, digits=3))
print("AUC:", round(roc_auc_score(y_test, test_proba), 4))

test accuracy: 0.8401 | random / always-'no link' baseline: 0.5
[[815 142]
 [164 793]]
              precision    recall  f1-score   support

           0      0.832     0.852     0.842       957
           1      0.848     0.829     0.838       957

    accuracy                          0.840      1914
   macro avg      0.840     0.840     0.840      1914
weighted avg      0.840     0.840     0.840      1914

AUC: 0.8845


*Reflect (Lab 5 Q7):* what does this accuracy mean next to the 50% baselines? A random classifier and a classifier that always predicts "no link" both score about 50% on a 50/50 test set. How far above that are you? Remember also that a real network has hundreds of non-links for every link, so accuracy on this balanced test would not carry over to a real deployment (precision and recall would matter more).

## 9. TODO: ideas to improve performance

Try these one at a time. For each: write down the hypothesis, change one thing, rebuild the tables with the same seed, and judge by the noise level from 6e (differences under about 0.015 are not evidence). Use validation only. Look at 6h first: are the errors mostly real links between different communities, or non-links inside a community?

1. **Separate real from fake pairs *inside* a community.** `same_community` cannot do this. Edge probability often grows with popularity, so try a log-degree product and an interaction with `same_community`.
2. **Graded community affinity.** Instead of yes/no, the fraction of u's neighbours that lie in v's community (and vice versa). Compute it on the feature graph (`H` or `T`) so there is no leakage.
3. **Stability of community detection.** `same_community` is your key feature. Run Louvain with several seeds and compare partitions (`sklearn.metrics.adjusted_rand_score`). Also try the Louvain `resolution` parameter.
4. **Retrain on all visible edges.** After all choices are fixed, build training features from all 6,377 edges with more folds (for example 10), refit, and predict with features from `G`. More positives, and a graph closer to the one used for the real pairs.
5. **Decision threshold.** Check on validation whether a cutoff other than 0.5 does better.
6. **Harder negatives.** Random non-links are easy. Also sample non-links at distance 2 so the model must tell friends-of-friends apart.
7. **Negative ratio.** Try 2 or 3 negatives per positive in training (keep evaluation 50/50), and check what changes.
8. **Node embeddings** (spectral or node2vec). Powerful but needs the same leakage care as `ccpa`; probably only worth a mention in the report.

In [126]:
# TODO: idea 1 - log-degree product and interaction (add to simple_features, then rebuild section 5)
# TODO: idea 2 - community affinity feature (add a function like community_features)
# TODO: idea 3 - community stability across seeds
# from sklearn.metrics import adjusted_rand_score
# parts = [nx.community.louvain_communities(T, seed=s) for s in range(5)]
# TODO: idea 5 - threshold search on validation probabilities
# for t in np.arange(0.3, 0.71, 0.05):
#     print(round(t, 2), accuracy_score(y_val, (proba >= t).astype(int)))

## 10. Notes for the report

**Technical solution**
- *Features:* similarity measures (NetworkX), degrees, shared attribute (homophily), same Louvain community. Final model uses `same_community`, `ccpa`, `same_attribute`.
- *Data preparation:* 70/15/15 split of visible edges, 1:1 random negatives checked against the full graph, training features from 5 folds so no positive sees its own edge, validation/test positives absent from `T`.
- *Model selection:* logistic regression (scaled) vs random forest, tied within noise, so the simpler model was chosen. Hyperparameters tuned by cross-validation on training data only.
- *Final results:* validation and test accuracy vs the 0.5 baseline; confusion matrix; AUC.
- *Limitations:* noise of about 0.015 in validation accuracy, density mismatch between training graphs and the graph used for the real pairs, balanced 50/50 evaluation, test set seen once earlier in development, network appears synthetic with 6 planted groups.

**Story of the development (each step has a reason)**
1. Baseline logistic regression: train 1.00 vs val 0.50. A perfect score on a hard task signalled a leak.
2. Traced it to `ccpa` (value 360 only when the edge is visible), fixed by fold-based training features.
3. Scaling, coefficient reading, feature ablation: attribute and closeness mattered, many features were duplicates.
4. Class-pair encoding and random forest: no improvement (negative results worth reporting).
5. Visualised the network: communities match attributes, so `same_community` added a large gain.

**Ethics (Part 2) hooks from this analysis**
- Removing `same_attribute` hurt accuracy, and communities match attributes: even a model that never sees a sensitive attribute can reveal it through who a person connects to (privacy).
- Predicting links by closeness reinforces existing clusters (fairness, filter bubbles).
- A model this simple is easy to explain (transparency), but the community partition itself is a design choice that is hard to justify to an individual.